# XLK Data Feasibility Test

This notebook assesses whether daily XLK data can support the proposed next-week high-volatility classification task. It examines data availability, constructs one historical volatility feature and a forward-looking realised-volatility target, and applies a provisional chronological split. No machine-learning model is trained.

In [ ]:
from pathlib import Path
import os

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.venv' / '.matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.venv' / 'cache'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

TICKER = 'XLK'
START_DATE = '2000-01-01'
END_DATE = '2026-07-01'
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'xlk_daily_2000_2026.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xlk_feasibility_dataset.csv'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feasibility_summary.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'

sns.set_theme(style='whitegrid')
print(f'Confirmed project directory: {PROJECT_ROOT}')
print(f'Data request: {TICKER}, daily observations from {START_DATE} to {END_DATE} (exclusive end date).')

## 1. Download and raw-data audit

The requested daily series is downloaded once and saved without altering the returned rows, index or columns. Any `MultiIndex` column structure is handled explicitly for subsequent analysis. Unexpected columns are reported and retained in the raw file.

In [ ]:
raw_download = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False,
)

if raw_download.empty:
    raise RuntimeError('The XLK download returned no observations; the feasibility test cannot continue.')

raw_download.to_csv(RAW_PATH)

if isinstance(raw_download.columns, pd.MultiIndex):
    ticker_levels = [
        level for level in range(raw_download.columns.nlevels)
        if set(raw_download.columns.get_level_values(level).astype(str)) == {TICKER}
    ]
    if len(ticker_levels) != 1:
        raise RuntimeError(
            f'Unable to identify one unambiguous ticker level in MultiIndex columns: {raw_download.columns.tolist()}'
        )
    analysis_data = raw_download.xs(TICKER, axis=1, level=ticker_levels[0], drop_level=True).copy()
    column_structure = f'MultiIndex columns detected; ticker level {ticker_levels[0]} was removed for analysis only.'
else:
    analysis_data = raw_download.copy()
    column_structure = 'Single-level columns detected; no structural conversion was required.'

if isinstance(analysis_data.columns, pd.MultiIndex):
    raise RuntimeError(f'Unexpected nested column structure remains: {analysis_data.columns.tolist()}')

expected_columns = {'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'}
observed_columns = set(analysis_data.columns.astype(str))
unexpected_columns = sorted(observed_columns - expected_columns)
missing_expected_columns = sorted(expected_columns - observed_columns)

if 'Adj Close' not in analysis_data.columns:
    raise RuntimeError(
        f'Adjusted Close is unavailable. Observed columns: {analysis_data.columns.tolist()}. No substitute will be used.'
    )

analysis_data.index = pd.DatetimeIndex(analysis_data.index, name='Date')
duplicate_dates = int(analysis_data.index.duplicated(keep=False).sum())
missing_by_column = analysis_data.isna().sum()

print(column_structure)
print(f'First observation date: {analysis_data.index.min().date()}')
print(f'Last observation date: {analysis_data.index.max().date()}')
print(f'Number of rows: {len(analysis_data):,}')
print(f'Column names: {analysis_data.columns.tolist()}')
print('Data types:')
print(analysis_data.dtypes.to_string())
print(f'Duplicate dates: {duplicate_dates}')
print('Missing values by column:')
print(missing_by_column.to_string())
print('Unexpected columns retained in the raw file:', unexpected_columns if unexpected_columns else 'None')
print('Missing expected columns:', missing_expected_columns if missing_expected_columns else 'None')

## 2. Returns, historical volatility and future realised volatility

Daily logarithmic returns use Adjusted Close. The annualised 20-trading-day rolling volatility at date *t* uses information available on or before *t*. The future five-day realised volatility uses precisely the five returns from *t*+1 through *t*+5 and excludes the return at *t*.

In [ ]:
assert analysis_data.index.is_monotonic_increasing and not analysis_data.index.has_duplicates, \
    'Dates must be strictly increasing and unique.'
assert analysis_data['Adj Close'].notna().all() and (analysis_data['Adj Close'] > 0).all(), \
    'All adjusted prices must be present and positive.'

data = analysis_data.copy()
data['log_return'] = np.log(data['Adj Close'] / data['Adj Close'].shift(1))
data['volatility_20d'] = data['log_return'].rolling(window=20, min_periods=20).std(ddof=1) * np.sqrt(252)

future_returns = pd.concat(
    {f't+{horizon}': data['log_return'].shift(-horizon) for horizon in range(1, 6)},
    axis=1,
)
complete_future_window = future_returns.notna().all(axis=1)
data['future_rv_5d'] = np.sqrt(future_returns.pow(2).sum(axis=1, min_count=5))

assert data['future_rv_5d'].notna().equals(complete_future_window), \
    'A valid future target must require all five future returns.'
assert data['future_rv_5d'].tail(5).isna().all(), \
    'The final five raw observations must not have future realised volatility.'

print('Return and volatility construction completed without using future information in the historical feature.')
print('Missing logarithmic returns:', int(data['log_return'].isna().sum()))
print('Missing 20-day volatility values:', int(data['volatility_20d'].isna().sum()))
print('Missing future five-day realised-volatility values:', int(data['future_rv_5d'].isna().sum()))

## 3. Provisional chronological split and target

Rows lacking the required feature or future target are excluded. The first 80 per cent of valid observations define the provisional training period, and the remaining observations define the test period. Because each target uses returns from *t*+1 through *t*+5, the final five provisional training observations have target windows that reach into the test period. These observations are purged before threshold estimation, creating a five-trading-day gap without moving the first test date. A single 75th-percentile threshold estimated from the final purged training sample is applied unchanged to both final samples. Adjacent daily targets still overlap within each sample; this is acceptable for the initial rolling-forecast design but must be considered when the final validation strategy is specified.

In [ ]:
required_columns = ['Adj Close', 'log_return', 'volatility_20d', 'future_rv_5d']
valid_data = data.dropna(subset=required_columns).copy()
valid_data = valid_data.sort_index()

split_position = int(np.floor(0.80 * len(valid_data)))
if split_position <= 0 or split_position >= len(valid_data):
    raise RuntimeError(f'The provisional 80/20 split is not feasible for {len(valid_data)} valid observations.')

provisional_training_index = valid_data.index[:split_position]
test_index = valid_data.index[split_position:]
original_split_date = test_index[0]
split_date = original_split_date
purge_length = 5
purged_index = provisional_training_index[-purge_length:]
training_index = provisional_training_index[:-purge_length]

training_target_end_dates = pd.DatetimeIndex([
    data.index[data.index.get_loc(date) + 5] for date in training_index
])
purged_target_end_dates = pd.DatetimeIndex([
    data.index[data.index.get_loc(date) + 5] for date in purged_index
])

assert len(purged_index) == 5, 'Exactly five observations must be purged.'
assert test_index[0] == original_split_date, 'The first test observation date must remain unchanged.'
assert (training_target_end_dates < split_date).all(), \
    'No training target window may include a return dated on or after the first test observation.'
assert (purged_target_end_dates >= split_date).all(), \
    'Each purged observation must have a target window that reaches into the test period.'

threshold_from_final_training = float(valid_data.loc[training_index, 'future_rv_5d'].quantile(0.75))
threshold = threshold_from_final_training
assert np.isclose(threshold, valid_data.loc[training_index, 'future_rv_5d'].quantile(0.75)), \
    'The threshold must be calculated using the final purged training sample only.'

final_data = pd.concat([valid_data.loc[training_index], valid_data.loc[test_index]]).copy()
final_data['sample_period'] = ['train'] * len(training_index) + ['test'] * len(test_index)
final_data['high_volatility'] = (final_data['future_rv_5d'] > threshold).astype(int)

expected_training_labels = (final_data.loc[training_index, 'future_rv_5d'] > threshold).astype(int)
expected_test_labels = (final_data.loc[test_index, 'future_rv_5d'] > threshold).astype(int)
assert final_data.index.is_monotonic_increasing and not final_data.index.has_duplicates, \
    'Final modelling dates must be strictly increasing and unique.'
assert set(final_data['high_volatility'].unique()).issubset({0, 1}), \
    'The high-volatility target must contain only zero and one.'
assert final_data.loc[training_index, 'high_volatility'].equals(expected_training_labels), \
    'The training-derived threshold must be applied to the final training sample.'
assert final_data.loc[test_index, 'high_volatility'].equals(expected_test_labels), \
    'The same training-derived threshold must be applied to the test sample.'
assert final_data.index.intersection(purged_index).empty, \
    'Purged observations must not be classified as training or test observations.'
assert (final_data.loc[training_index, 'sample_period'] == 'train').all(), \
    'The training-period labels do not match the purged chronological split.'
assert (final_data.loc[test_index, 'sample_period'] == 'test').all(), \
    'The test-period labels do not match the chronological split.'

def class_summary(frame):
    counts = frame['high_volatility'].value_counts().reindex([0, 1], fill_value=0)
    proportions = frame['high_volatility'].value_counts(normalize=True).reindex([0, 1], fill_value=0.0)
    return pd.DataFrame({'count': counts, 'proportion': proportions}).rename_axis('high_volatility')

full_classes = class_summary(final_data)
train_classes = class_summary(final_data.loc[training_index])
test_classes = class_summary(final_data.loc[test_index])

print(f'Original provisional split date (first test observation): {split_date.date()}')
print(f'Provisional training sample size before purging: {len(provisional_training_index):,}')
print(f'Final training sample size after purging: {len(training_index):,}')
print(f'Unchanged test sample size: {len(test_index):,}')
print(f'Number of purged observations: {len(purged_index)}')
print('Purged observation dates:', ', '.join(date.date().isoformat() for date in purged_index))
print(f'Recalculated training-derived 75th-percentile threshold: {threshold:.10f}')
for label, summary in [('Final modelling sample', full_classes), ('Final training sample', train_classes), ('Unchanged test sample', test_classes)]:
    print(f'\n{label} class counts and proportions:')
    print(summary.to_string(float_format=lambda value: f'{value:.6f}'))

## 4. Figures

The figures retain zero as the lower bound for non-negative price, volatility and proportion axes. Training and test target proportions are displayed separately.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data.index, data['Adj Close'], color='#1f77b4', linewidth=1.2)
ax.set_title('XLK Adjusted Close Price')
ax.set_xlabel('Date')
ax.set_ylabel('Adjusted close price (US dollars)')
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xlk_adjusted_close.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data.index, data['volatility_20d'], color='#d62728', linewidth=1.0)
ax.set_title('XLK Annualised 20-Trading-Day Rolling Volatility')
ax.set_xlabel('Date')
ax.set_ylabel('Annualised volatility')
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xlk_rolling_volatility.png', dpi=300, bbox_inches='tight')
plt.show()

proportion_plot = pd.DataFrame({
    'Training period': train_classes['proportion'],
    'Test period': test_classes['proportion'],
}).T
proportion_plot.columns = ['Normal-volatility class (0)', 'High-volatility class (1)']

fig, ax = plt.subplots(figsize=(10, 6))
proportion_plot.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518'], width=0.72)
ax.set_title('XLK Target-Class Proportions by Sample Period')
ax.set_xlabel('Sample period')
ax.set_ylabel('Proportion of observations')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Target class')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xlk_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved three high-resolution feasibility figures.')

## 5. Modelling dataframe, summary and final quality checks

The modelling dataframe contains only final training and test observations; the five purged observations are excluded. The concise summary records the data range, provisional and final sample sizes, purge dates, unchanged split, recalculated threshold, updated class results and raw-data quality findings.

In [ ]:
modelling_data = final_data[[
    'Adj Close', 'log_return', 'volatility_20d', 'future_rv_5d', 'high_volatility', 'sample_period'
]].rename(columns={'Adj Close': 'Adjusted Close'}).reset_index()

assert not modelling_data.isna().any().any(), \
    'The processed modelling dataframe must contain no missing values.'
assert modelling_data['Date'].is_monotonic_increasing and modelling_data['Date'].is_unique, \
    'Processed dates must be strictly increasing and unique.'
assert (modelling_data['Adjusted Close'] > 0).all(), \
    'Processed adjusted prices must be positive.'
assert set(modelling_data['high_volatility'].unique()) == {0, 1}, \
    'Both binary target classes must be present in the modelling dataframe.'
assert modelling_data.set_index('Date').index.intersection(purged_index).empty, \
    'The processed modelling dataframe must contain no purged observations.'

modelling_data.to_csv(PROCESSED_PATH, index=False)

missing_finding = '; '.join(f'{column}: {int(count)}' for column, count in missing_by_column.items())
summary_rows = [
    ('ticker', TICKER),
    ('first_observation_date', analysis_data.index.min().date().isoformat()),
    ('last_observation_date', analysis_data.index.max().date().isoformat()),
    ('raw_sample_size', len(analysis_data)),
    ('valid_sample_size_before_purging', len(valid_data)),
    ('final_modelling_sample_size', len(final_data)),
    ('original_split_date_first_test_observation', split_date.date().isoformat()),
    ('provisional_training_sample_size', len(provisional_training_index)),
    ('final_training_sample_size_after_purging', len(training_index)),
    ('unchanged_test_sample_size', len(test_index)),
    ('purged_observation_count', len(purged_index)),
    ('purged_observation_dates', '; '.join(date.date().isoformat() for date in purged_index)),
    ('training_75th_percentile_threshold', threshold),
    ('final_sample_class_0_count', int(full_classes.loc[0, 'count'])),
    ('final_sample_class_0_proportion', float(full_classes.loc[0, 'proportion'])),
    ('final_sample_class_1_count', int(full_classes.loc[1, 'count'])),
    ('final_sample_class_1_proportion', float(full_classes.loc[1, 'proportion'])),
    ('training_class_0_count', int(train_classes.loc[0, 'count'])),
    ('training_class_0_proportion', float(train_classes.loc[0, 'proportion'])),
    ('training_class_1_count', int(train_classes.loc[1, 'count'])),
    ('training_class_1_proportion', float(train_classes.loc[1, 'proportion'])),
    ('test_class_0_count', int(test_classes.loc[0, 'count'])),
    ('test_class_0_proportion', float(test_classes.loc[0, 'proportion'])),
    ('test_class_1_count', int(test_classes.loc[1, 'count'])),
    ('test_class_1_proportion', float(test_classes.loc[1, 'proportion'])),
    ('duplicate_dates', duplicate_dates),
    ('missing_values_by_column', missing_finding),
]
summary_table = pd.DataFrame(summary_rows, columns=['metric', 'value'])
summary_table.to_csv(SUMMARY_PATH, index=False)

print(f'Processed modelling dataframe saved with {len(modelling_data):,} complete observations.')
print(f'Raw data file: {RAW_PATH.relative_to(PROJECT_ROOT)}')
print(f'Processed data file: {PROCESSED_PATH.relative_to(PROJECT_ROOT)}')
print(f'Summary table: {SUMMARY_PATH.relative_to(PROJECT_ROOT)}')
print('All specified quality checks passed.')
display(summary_table)

## Feasibility scope

This notebook establishes data and target feasibility only. The threshold and chronological split remain provisional, and no conclusion about predictive performance can be drawn until a separate modelling stage is designed and validated.